# Taller 4 - Entrenamiento con MLflow (Penguins)

Notebook que:
1. Carga datos raw desde PostgreSQL (dataset Penguins)
2. Procesa y guarda en data_procesada
3. Ejecuta 20+ experimentos con variación de hiperparámetros
4. Registra todo en MLflow

## 1. Configuración y carga de datos

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

PG_HOST = os.environ.get("POSTGRES_HOST", "postgres")
PG_PORT = os.environ.get("POSTGRES_PORT", "5432")
PG_USER = os.environ.get("POSTGRES_USER", "mlflow_user")
PG_PASS = os.environ.get("POSTGRES_PASSWORD", "mlflow_pass")
CONN_RAW = f"postgresql://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/mlflow_db"

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000"))
# Si el experimento existe con ubicación s3://, eliminarlo para usar proxy (mlflow-artifacts://)
try:
    exp = mlflow.get_experiment_by_name("penguins-rf")
    if exp and "s3://" in (exp.artifact_location or ""):
        mlflow.delete_experiment(exp.experiment_id)
        print("Experimento anterior eliminado (usaba S3 directo). Se creará uno nuevo con proxy.")
except Exception:
    pass
mlflow.set_experiment("penguins-rf")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1773832067001, experiment_id='1', last_update_time=1773832067001, lifecycle_stage='active', name='penguins-rf', tags={}>

In [2]:
# Cargar datos raw desde PostgreSQL (o desde CSV si la tabla está vacía)
engine = create_engine(CONN_RAW)
try:
    df_raw = pd.read_sql("SELECT * FROM taller_data.data_raw", con=engine)
    if len(df_raw) == 0:
        raise ValueError("Tabla vacía")
except Exception:
    # Cargar desde CSV y guardar en PostgreSQL
    csv_path = "/home/jovyan/data/penguins.csv"
    df_raw = pd.read_csv(csv_path)
    df_raw.to_sql("data_raw", engine, if_exists="replace", index=False, schema="taller_data")
    print(f"Cargados {len(df_raw)} registros desde CSV a data_raw")
else:
    print(f"Registros en data_raw: {len(df_raw)}")

Registros en data_raw: 344


## 2. Procesamiento y guardado en data_procesada

In [3]:
# Procesar: eliminar nulos, codificar categóricas
df = df_raw.dropna()
le_island = LabelEncoder()
le_sex = LabelEncoder()
le_species = LabelEncoder()
df["island_encoded"] = le_island.fit_transform(df["island"].astype(str))
df["sex_encoded"] = le_sex.fit_transform(df["sex"].astype(str))
df["species_encoded"] = le_species.fit_transform(df["species"].astype(str))

cols_procesada = [
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g",
    "island_encoded", "sex_encoded", "year", "species_encoded"
]
df_proc = df[cols_procesada]

# Guardar en PostgreSQL
df_proc.to_sql("data_procesada", engine, if_exists="replace", index=False, schema="taller_data")
print(f"Guardados {len(df_proc)} registros en data_procesada")

Guardados 333 registros en data_procesada


/tmp/ipykernel_1813/559520336.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["island_encoded"] = le_island.fit_transform(df["island"].astype(str))
/tmp/ipykernel_1813/559520336.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["sex_encoded"] = le_sex.fit_transform(df["sex"].astype(str))
/tmp/ipykernel_1813/559520336.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the doc

## 3. Entrenamiento con 20+ experimentos (hiperparámetros)

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

X = df_proc.drop("species_encoded", axis=1)
y = df_proc["species_encoded"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Hiperparámetros a explorar (20+ combinaciones)
n_estimators_list = [50, 100, 150, 200]
max_depth_list = [5, 10, 15, 20, None]
min_samples_split_list = [2, 5, 10]

run_count = 0
for n_est in n_estimators_list:
    for max_d in max_depth_list:
        for min_ss in min_samples_split_list:
            if run_count >= 20:
                break
            with mlflow.start_run():
                model = RandomForestClassifier(
                    n_estimators=n_est,
                    max_depth=max_d,
                    min_samples_split=min_ss,
                    random_state=42
                )
                model.fit(X_train, y_train)
                pred = model.predict(X_test)
                acc = accuracy_score(y_test, pred)
                f1 = f1_score(y_test, pred, average="weighted")
                mlflow.log_param("n_estimators", n_est)
                mlflow.log_param("max_depth", max_d if max_d else -1)
                mlflow.log_param("min_samples_split", min_ss)
                mlflow.log_metric("accuracy", acc)
                mlflow.log_metric("f1_weighted", f1)
                signature = infer_signature(X_train, model.predict(X_train))
                mlflow.sklearn.log_model(model, "model", registered_model_name="PenguinsRF", signature=signature)
            run_count += 1
        if run_count >= 20:
            break
    if run_count >= 20:
        break

print(f"Ejecutados {run_count} experimentos en MLflow")

/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'PenguinsRF' already exists. Creating a new version of this model...
2026/03/18 11:27:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PenguinsRF, versi

Ejecutados 20 experimentos en MLflow


In [5]:
# Promover el mejor modelo a Production en MLflow
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = mlflow.get_experiment_by_name("penguins-rf")
if experiment:
    runs = client.search_runs(experiment_ids=[experiment.experiment_id], order_by=["metrics.accuracy DESC"])
    if runs:
        best_run = runs[0]
        for mv in client.search_model_versions(f"run_id='{best_run.info.run_id}'"):
            client.transition_model_version_stage("PenguinsRF", mv.version, "Production")
            print(f"Modelo v{mv.version} (run {best_run.info.run_id}) promovido a Production")
            break

Modelo v39 (run d29616fa9a974db6a74b8c3ff4e5841a) promovido a Production


/tmp/ipykernel_1813/824107704.py:11: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage("PenguinsRF", mv.version, "Production")
